# Calibration gradients & the Hessian

Calibration recovers the detector's optical parameters (scattering, absorption, reflection, QE)
by **descending a smooth loss**. This notebook opens up *why* that works — the calibration
analog of `track_gradients`:

1. **Gradient** — the loss landscape over the optical parameters (smooth, minimum at truth).
2. **Hessian** — the curvature = Fisher information: which parameters are well-constrained,
   which are degenerate, and the achievable uncertainty (Cramér–Rao bound).
3. **Before / after** — the state of the fit before it starts vs after it converges.

Everything is library calls: `lucid.gradient_analysis` (sweeps), `lucid.losses.WC_smooth_loss`,
and `lucid.fitting.{build_calibration_problem, fit, crb}`.

> Note: the sweep cells visualize `WC_smooth_loss`; the `fit()`/`crb()` calls use `lucid.fitting`'s own √-MSE Gauss-Newton objective internally — two related but distinct loss forms.


In [ ]:
import sys; sys.path.append('..')
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.sources import laser_source, isotropic_source
from lucid.detector_params import DetectorParams
from lucid.losses import WC_smooth_loss
from lucid.fitting import build_calibration_problem, fit, crb
from lucid.gradient_analysis import SweepParam, sweep_1d, sweep_2d, plot_sweep_1d, plot_sweep_2d_single

from tutorial_profile import PROFILE as P
P.describe()   # laptop mode by default; set LUCID_TUTORIAL_MODE=full for full fidelity

GEOM = '../config/SK_like_geom_config.json'
det = generate_detector(GEOM); NS = len(det.all_points); pts = jnp.asarray(det.all_points)
top, bot, R = det.H/2 - .1, -det.H/2 + .1, det.r

# truth optical parameters + a diverse set of calibration light sources
dp = DetectorParams.from_flat(scatter_length=70., mie_scatter_length=3000., g=0.9,
     wall_reflection_rate=.2, sensor_reflection_rate=.2, absorption_length=60., qe=0.07,
     qe_corrections=jnp.ones(NS))
sources = [laser_source(position=[0, 0, top], direction=[0, 0, -1], intensity=1e6),
           laser_source(position=[0, 0, bot], direction=[0, 0,  1], intensity=1e6),
           laser_source(position=[R-.1, 0, 0], direction=[-1, 0, 0], intensity=1e6),
           isotropic_source(position=[0, 0, 0], intensity=1e6)]
sim = setup_event_simulator(GEOM, P.n_photons_cal, temperature=None, K=P.K_water, is_calibration=True,
                            hit_mode='aggregated', wavelength_mode=False,
                            **P.grid)
FIELDS = ['g', 'scatter_length', 'mie_scatter_length', 'absorption_length',
          'wall_reflection_rate', 'sensor_reflection_rate', 'qe']
prob = build_calibration_problem(sim, sources, dp, FIELDS, key=jax.random.PRNGKey(1))
print(f'{NS} PMTs | {len(sources)} calibration sources | {len(FIELDS)} global parameters')

## 1. Gradient — the loss landscape
Sweep each optical parameter around truth and plot the loss (top) with its **automatic-
differentiation gradient** vs the finite-difference gradient (bottom). The loss is smooth and
minimised at truth; AD matches FD.

In [ ]:
# one strong laser defines a landscape we can sweep quickly
src = sources[0]
sim_truth = setup_event_simulator(GEOM, P.n_photons_cal, temperature=None, K=P.K_water, is_calibration=True,
              hit_mode='aggregated', wavelength_mode=False, default_detector_params=dp,
              **P.grid)
true_data = jax.lax.stop_gradient(sim_truth(src, jax.random.PRNGKey(9)))

@jax.jit
def loss_and_grad(p):
    def L(x):
        return WC_smooth_loss(pts, *true_data, *sim(src, x, jax.random.PRNGKey(3)),
                              lambda_poisson=1.0, lambda_time=0.0, tau=0.5)
    return jax.value_and_grad(L)(p)
_ = loss_and_grad(dp)   # warm up the JIT

sweeps = [SweepParam('Scatter Length',    'scatter_length',       half_width=20., unit='m', min_val=0.001, grad_scale=100., num_points=P.scan_1d),
          SweepParam('Absorption Length', 'absorption_length',    half_width=40., unit='m', min_val=25.,   grad_scale=100., num_points=P.scan_1d),
          SweepParam('Wall Reflection',   'wall_reflection_rate', half_width=0.15, min_val=0., max_val=1., grad_scale=0.1, num_points=P.scan_1d),
          SweepParam('QE',                'qe',                   half_width=0.03, min_val=0.001,          grad_scale=0.1, num_points=P.scan_1d)]
res1d = sweep_1d(loss_and_grad, dp, sweeps)
plot_sweep_1d(res1d, title='Calibration loss & gradient — 1D sweeps')

### 2D landscape — a parameter degeneracy
Absorption length and QE trade off (both scale the detected light), so their 2D loss surface
has a **valley** — the calibration analog of a degenerate direction.

In [ ]:
res2d = sweep_2d(loss_and_grad, dp, sweeps[1], sweeps[3], num_points=P.scan_2d)  # absorption x QE
plot_sweep_2d_single(res2d, title='Absorption length x QE — degeneracy valley')

## 2. Hessian — curvature, correlations, uncertainty
The curvature of the loss at the optimum is the **Fisher information** `F` (the expected Hessian
of the Poisson NLL), computed here with source diversity by `crb`. Its inverse is the parameter
**covariance** — the achievable uncertainty (Cramér–Rao bound) and the correlations between
parameters. `crb` returns `fisher`, `cov`, and the fractional `sigma`.

In [ ]:
c = crb(prob['source_models'], prob['theta_true'], NS)
F, cov, sigma = c['fisher'], c['cov'], c['sigma']
Dg = np.sqrt(np.diag(cov)); corr = cov / np.outer(Dg, Dg)              # parameter correlation
Fn = F / np.sqrt(np.outer(np.diag(F), np.diag(F)))                     # normalised Fisher (coupling)
ev = np.linalg.eigvalsh(F); cond = ev.max() / ev.min()

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
im0 = ax[0].imshow(Fn, cmap='viridis', vmin=-1, vmax=1)
ax[0].set_title('normalised Fisher (Hessian) — coupling'); plt.colorbar(im0, ax=ax[0], fraction=.046)
im1 = ax[1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax[1].set_title(f'parameter correlation (F$^{{-1}}$)'); plt.colorbar(im1, ax=ax[1], fraction=.046)
for a in ax[:2]:
    a.set_xticks(range(len(FIELDS))); a.set_yticks(range(len(FIELDS)))
    a.set_xticklabels(FIELDS, rotation=40, ha='right', fontsize=7); a.set_yticklabels(FIELDS, fontsize=7)
ax[2].bar(range(len(FIELDS)), np.array(sigma)*100, color='#c0392b', alpha=.8)
ax[2].set_xticks(range(len(FIELDS))); ax[2].set_xticklabels(FIELDS, rotation=40, ha='right', fontsize=7)
ax[2].set_ylabel('%'); ax[2].set_title('CRB (fractional 1$\\sigma$) per parameter'); ax[2].grid(alpha=.3, axis='y')
fig.tight_layout(); plt.show()

iu = np.triu_indices(len(FIELDS), 1); k = int(np.argmax(np.abs(corr[iu])))
print(f'Fisher condition number: {cond:.1e}  (spread of well- vs poorly-constrained directions)')
print(f'strongest degeneracy: {FIELDS[iu[0][k]]} <-> {FIELDS[iu[1][k]]} = {corr[iu][k]:+.2f}')

## 3. Before and after the fit
**Before:** start from a +15% perturbed guess — off truth, high loss, on the slope of the
landscape above. **After:** `fit` descends to truth; the Hessian above gives the post-fit
uncertainty. We compare every parameter to truth and to its CRB, and watch the fit converge.

In [ ]:
start = prob['theta_true'] + np.random.default_rng(0).uniform(-.15, .15, prob['theta_true'].shape)
res = fit(prob['source_models'], prob['truth_charge'], start, NS, steps=P.fit_steps, refresh=15, nb_h=2)
truth = np.exp(prob['theta_true'])

print(f'{"param":22s}{"truth":>10s}{"start":>10s}{"fit":>10s}{"err":>8s}{"CRB":>8s}')
for i, f in enumerate(FIELDS):
    e = res['theta'][i]/truth[i] - 1
    print(f'{f:22s}{truth[i]:10.3f}{np.exp(start[i]):10.3f}{res["theta"][i]:10.3f}{e:+7.1%}{sigma[i]:8.1%}')

# convergence: per-parameter fractional error vs GN iteration (from fit history)
hist = np.asarray([res['history'][s] for s in range(len(res['history']))])   # (steps, nparam), linear
fig, ax = plt.subplots(figsize=(9, 4))
for i, f in enumerate(FIELDS):
    ax.plot(100*(hist[:, i]/truth[i] - 1), label=f)
ax.axhline(0, color='k', lw=.7); ax.set(xlabel='GN iteration', ylabel='fractional error (%)',
          title='before -> after: parameters converge to truth')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3); ax.set_ylim(-20, 20); fig.tight_layout(); plt.show()

# before/after Hessian check: the Fisher is stable across the basin (near-quadratic loss)
c0 = crb(prob['source_models'], start, NS)
print(f'\nCRB is stable start->fit (near-quadratic basin): '
      f'e.g. scatter_length sigma {c0["sigma"][1]:.1%} (start) vs {sigma[1]:.1%} (truth)')

## Takeaways
- The calibration loss is a **smooth** function of the optical parameters — its gradient points
  to truth (AD == finite-difference), so gradient descent recovers them.
- The **Hessian / Fisher** quantifies the calibration: the condition number and correlation
  matrix reveal which parameters are well-constrained vs **degenerate** (e.g. absorption↔QE),
  and its inverse gives the **CRB** — the best achievable uncertainty. Source diversity is what
  keeps this matrix well-conditioned.
- **Before → after:** from a +15% start the fit converges to truth within the CRB, and the
  Fisher is stable across the basin, so the post-fit uncertainty estimate is reliable.

See `calibration_optimization` for the fit-vs-CRB summary, and `track_gradients` for the
reconstruction analog of this notebook.